# Election Prediction 2026 — Interactive Notebook

**Run this notebook cell-by-cell to explore data and generate predictions.**

## Quick-Start Checklist
1. Place your ECI Excel files in `data/raw/` (naming: `TamilNadu_2021_DetailedResults.xlsx`)
2. Run cells in order
3. When flagged constituencies appear, fill in `outputs/review/swing_seats_review.xlsx`
4. Re-run the final cell to regenerate with your overrides

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.chdir('..')  # Run from project root

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Setup complete. CWD:', os.getcwd())

## Stage 1 — Load and Inspect Raw Data

In [ ]:
from src.pipeline.ingestion import ingest_data

df_raw = ingest_data('data/raw/')
print(f'Raw data shape: {df_raw.shape}')
print(f'\nStates found: {df_raw["state"].unique()}')
print(f'Years found:  {sorted(df_raw["year"].dropna().unique())}')
df_raw.head(3)

In [ ]:
# Constituency count per state-year
df_raw.groupby(['state','year'])['constituency_name'].nunique().unstack()

## Stage 2 — Clean Data

In [ ]:
from src.pipeline.cleaning import clean_data

df_clean = clean_data(df_raw)
print(f'Clean data: {df_clean.shape}')

# Spot-check Tamil Nadu 2021
tn_2021 = df_clean[(df_clean['state']=='Tamil Nadu') & (df_clean['year']==2021)]
print(f'\nTN 2021 constituencies: {tn_2021["constituency_name"].nunique()}')
print(f'TN 2021 winners:\n{tn_2021[tn_2021["is_winner"]==1]["party"].value_counts().head(10)}')

In [ ]:
# Visualise party-wise seat wins over time (Tamil Nadu)
import matplotlib.pyplot as plt

tn_wins = (
    df_clean[(df_clean['state']=='Tamil Nadu') & (df_clean['is_winner']==1)]
    .groupby(['year','party'])
    .size()
    .reset_index(name='seats')
)
top_parties = tn_wins.groupby('party')['seats'].sum().nlargest(5).index
tn_top = tn_wins[tn_wins['party'].isin(top_parties)]

fig, ax = plt.subplots(figsize=(10,5))
for party, grp in tn_top.groupby('party'):
    ax.plot(grp['year'], grp['seats'], marker='o', label=party)
ax.set_title('Tamil Nadu — Seats Won by Party (Historical)')
ax.set_xlabel('Year')
ax.set_ylabel('Seats')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/reports/tn_historical_seats.png', dpi=150)
plt.show()

## Stage 3 — Feature Engineering

In [ ]:
from src.features.base_features import build_features
from src.features.alliance_inference import build_alliance_features

TARGET_YEAR = 2026

features = build_features(df_clean, cutoff_year=TARGET_YEAR)
features = build_alliance_features(df_clean, features, TARGET_YEAR)

print(f'Feature matrix: {features.shape}')
print(f'\nFeature columns:')
print([c for c in features.columns if c not in ['state','constituency_name','party','const_state_key']])

In [ ]:
# Check features for a specific constituency
CONSTITUENCY = 'ariyalur'  # Change this
STATE = 'Tamil Nadu'

sample = features[
    (features['state']==STATE) & 
    (features['constituency_name']==CONSTITUENCY.lower())
].T
print(f'Features for {STATE} — {CONSTITUENCY}:')
display(sample)

## Stage 4 — Backtest (Validate on 2021 Data)

In [ ]:
from src.models.evaluator import ElectionEvaluator

# This tests: 'if we ran this model before 2021, how accurate would it be?'
evaluator = ElectionEvaluator()
backtest_metrics = evaluator.backtest(
    df_clean,
    target_year=2021,
    states=['Tamil Nadu']   # Start with TN, your strongest state
)

print(f"\nBacktest Accuracy: {backtest_metrics.get('overall_accuracy',0)*100:.1f}%")
print(f"Correct: {backtest_metrics.get('correct_count',0)} / {backtest_metrics.get('total_count',0)}")
print(f"Vote Share MAE: {backtest_metrics.get('vs_mae', 'N/A')}")

## Stage 5 — Train Model for 2026

In [ ]:
from src.models.lgbm_model import ElectionLGBM

model = ElectionLGBM()
cv_metrics = model.train(
    features,
    df_clean,
    target_year=2026,
    tune_hyperparams=False  # Set True for better results (takes ~5 mins)
)
print('Training metrics:', cv_metrics)

In [ ]:
# SHAP feature importance
importance = model.explain(features)
importance.head(15)

## Stage 6 — Generate Predictions + Apply Rules

In [ ]:
from src.rules.rule_engine import RuleEngine

# -- Add your TN domain knowledge here --
TN_MANUAL_OVERRIDES = {
    # ('Tamil Nadu', 'constituency_name_lowercase'): 'PARTY',
    # Example: ('Tamil Nadu', 'coimbatore south'): 'BJP',
}

predictions = model.predict(features)
engine = RuleEngine()
predictions = engine.apply(predictions, features, manual_overrides=TN_MANUAL_OVERRIDES)

# Summary
winners = predictions[predictions['predicted_winner']==1]
print('Predicted seat counts by state:')
print(winners.groupby(['state','party']).size().unstack(fill_value=0))

## Stage 7 — Swing Scenarios

In [ ]:
from src.simulation.swing_simulator import SwingSimulator

simulator = SwingSimulator()
scenario_results = simulator.run_all_states(predictions)

# Seat projection table
seat_table = simulator.seat_projection_summary(scenario_results)
print('Seat projections across scenarios:')
display(seat_table.sort_values('state'))

In [ ]:
# Consensus blend (neutral 60%, optimistic 20%, pessimistic 20%)
consensus = simulator.build_consensus_prediction(scenario_results)
print('Consensus prediction seat counts:')
print(consensus[consensus['predicted_winner']==1].groupby(['state','party']).size().unstack(fill_value=0))

## Stage 8 — Human Review (Swing Seats)

In [ ]:
from src.human_loop.review_interface import HumanReviewInterface

hitl = HumanReviewInterface(margin_threshold=6.0)
safe_df, flagged_df = hitl.flag_for_review(predictions)

print(f'Flagged for review: {flagged_df["constituency_name"].nunique()} constituencies')
print(f'High confidence:    {safe_df["constituency_name"].nunique()} constituencies')

# Export the review sheet — open this and fill in your expert predictions!
review_path = hitl.export_review_sheet(flagged_df)
print(f'\n>> Review file: {review_path}')
print('>> Fill in EXPERT_OVERRIDE_WINNER column, then run the next cell')

In [ ]:
# After filling review file, run this to apply corrections
review_file = 'outputs/review/swing_seats_review.xlsx'
corrections = hitl.read_corrections(review_file)
print(f'Expert corrections loaded: {len(corrections)}')

if corrections:
    predictions = hitl.merge_corrections(predictions, corrections)
    print('Corrections applied!')
    winners_after = predictions[predictions['predicted_winner']==1]
    print(winners_after.groupby(['state','party']).size().unstack(fill_value=0))

## Stage 9 — Generate Submission Files

In [ ]:
from src.pipeline.output_generator import SubmissionGenerator, generate_methodology_note

generator = SubmissionGenerator()
submission_path = generator.generate_submission(
    predictions, 
    filename='final_predictions_v1.xlsx'
)

note_path = generate_methodology_note(cv_metrics)

print(f'\n✅ Submission file: {submission_path}')
print(f'✅ Methodology note: {note_path}')
print('\nRemember: You can submit up to 5 times. Submit now to lock in a timestamp!')

## Quick Constituency Lookup
Use this to inspect any specific constituency before deciding on a manual override.

In [ ]:
def inspect_constituency(state, constituency_name):
    """Show historical data + current prediction for any constituency."""
    const_lower = constituency_name.lower()
    
    print(f'=== {state} — {constituency_name.title()} ===')
    print()
    
    # Historical results
    hist = df_clean[
        (df_clean['state']==state) & 
        (df_clean['constituency_name']==const_lower)
    ].sort_values(['year','position'])
    
    print('Historical Results:')
    display(hist[['year','candidate_name','party','votes','vote_share','is_winner','margin_pct']]
            .query('position <= 3'))
    
    # Current prediction
    pred = predictions[
        (predictions['state']==state) & 
        (predictions['constituency_name']==const_lower)
    ].sort_values('predicted_vs', ascending=False)
    
    print('\n2026 Prediction (top 4):')
    display(pred[['party','predicted_vs','predicted_winner','confidence_level','rule_applied']].head(4))

# Example usage:
inspect_constituency('Tamil Nadu', 'ariyalur')